In [1]:
source('../../../R/preamble.R')

# Stickiness of 60,000 generations cells

In [2]:
cv_blank = 0.0830416666666667 # from the blanks_and_dilutions measurement (mean of 24 methanol-only wells)

well_to_cv_raw = read.csv('../../../datasets/adhesion/stickiness/cv.tsv', sep = '\t', check.names = F) %>% 
    pivot_longer(-y) %>%
    mutate(
        well = paste0(y, name),
        cv = value - cv_blank
    ) %>%
    select(well, cv)

In [3]:
well_to_cv_dilution = read.csv('../../../datasets/adhesion/stickiness/cv_blanks_and_dilutions.tsv', sep = '\t', check.names = F) %>% 
    pivot_longer(-y) %>%
    mutate(
        well = paste0(y, name),
        cv = (value - cv_blank) * 10 # 10-fold dilution
    ) %>%
    select(well, cv)

well_to_cv = rbind(
    well_to_cv_raw %>% filter(!well %in% well_to_cv_dilution$well),
    well_to_cv_dilution
    )

In [4]:
t_start = c(2000, 3000)
t_end = 13243.9

well_to_od =
read.csv('../../../datasets/adhesion/stickiness/growth.tsv', sep = '\t') %>% 
    pivot_longer(-t, names_to = 'well', values_to = 'od') %>%
    group_by(well) %>% 
    summarise(
        final_od = median(od[t == t_end]) - median(od[t >= t_start[1] & t <= t_start[2]])
    )

In [5]:
well_to_name = read.csv('../../../datasets/adhesion/stickiness/well_to_name.tsv', sep = '\t', check.names = F) %>% 
    pivot_longer(-y) %>%
    mutate(well = paste0(y, name)) %>%
    select(well, name = value)

In [6]:
stickiness = well_to_name %>%
    left_join(well_to_od) %>%
    left_join(well_to_cv) %>%
    mutate(strain_raw = substr(name, 1, 1), gen = as.integer(substr(name, 2, 3))) %>%
    mutate(
        strain = ifelse(strain_raw == 'H', 'E', strain_raw),
        s = cv / final_od
    )

wt_s = mean(stickiness[stickiness$strain == 'W',]$s)

# Duplicate the F06 measurements, that are part of both F and E's histories
stickiness = rbind(
    stickiness,
    stickiness %>% filter(name == 'F06') %>% mutate(strain = 'E'),
    stickiness %>% filter(name == 'W00') %>% mutate(strain = 'A'),
    stickiness %>% filter(name == 'W00') %>% mutate(strain = 'B'),
    stickiness %>% filter(name == 'W00') %>% mutate(strain = 'C'),
    stickiness %>% filter(name == 'W00') %>% mutate(strain = 'D'),
    stickiness %>% filter(name == 'W00') %>% mutate(strain = 'E'),
    stickiness %>% filter(name == 'W00') %>% mutate(strain = 'F')
    ) %>%
    mutate(v = s / wt_s) %>% 
    filter(strain != 'W')

Joining with `by = join_by(well)`


Joining with `by = join_by(well)`


In [7]:
write.tsv(stickiness, '../../../datasets/adhesion/stickiness/stickiness.tsv')